Definition and fitting function for single gaussians. 

Input args:

file_directory = folder path giving images
rows = number of rows to skip in text files
image_time = input from user
filename = filename for saved plot

Outputs: 
x_fit = peak fit
time_scatter = time array for colourbar + time series exponential fit (array) 
fwhm = full-width half max of peak fit (array)
fit_min = min from peak using 1/2 fwhm (array) 
fit_max = max " " " " " " (array)
x_raw = raw x data (array)
y_raw = raw y_data (array)

x_fit, x_raw, y_raw should be same length
time_scatter should be equal to number of files in folder. Ensure files are in time order before running. 



In [1]:
import numpy as np


In [2]:
#Fitting equation
def gaussian(x, A, mu, sigma):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2))


In [3]:
# Gaussian fitting function using data from a dataset (i.e. from each individual file) 
def fit_gaussian_from_dataset(x_data, y_data):
    """
    Fit a Gaussian curve to the provided dataset (x_data, y_data).
    Args:
    - x_data: Array-like, x values of the data
    - y_data: Array-like, y values of the data
    
    Returns:
    - A_fit: Amplitude of the fitted Gaussian
    - mu_fit: Mean (center) of the fitted Gaussian
    - sigma_fit: Standard deviation (width) of the fitted Gaussian
    - popt: Optimal parameters from curve fitting
    - pcov: Covariance matrix
    """
    # Initial guess for the parameters [A, mu, sigma]
    initial_guess = [max(y_data), np.mean(x_data), np.std(x_data)]

    try:
        # Fit the Gaussian curve to the data
        popt, pcov = curve_fit(gaussian, x_data, y_data, p0=initial_guess, maxfev=100000000)
        
        # Extract fitted parameters
        A_fit, mu_fit, sigma_fit = popt
        min_raw=np.min(x_data)
        mean_raw=np.mean(x_data)
        max_raw=np.max(x_data)
        # Print the fitted parameters for feedback
        #print(f"Fitted parameters:\nAmplitude: {A_fit}\nMean: {mu_fit}\nSigma: {sigma_fit}")
        #print(f"Raw data parameters are:\n Min = {min_raw}\n Mean = {mean_raw}\n Max = {max_raw}")
       
        # Return the fitted parameters and covariance matrix
        return A_fit, mu_fit, sigma_fit, popt, pcov, min_raw, mean_raw, max_raw

    except Exception as e:
        print(f"Error fitting Gaussian: {e}")
        return None, None, None, None, None

In [ ]:
#Fitting for single gaussian
def fit_gaussian_from_files(file_directory, rows, image_time, filename=None):
    min_scatter=[]
    mean_scatter=[]
    max_scatter=[]
    time_scatter=[]
    gaussian_scatter=[]
    fit_min = []
    fit_max = []
    
    t=-int(image_time)
    fig, ax = plt.subplots(1,1, sharey = True, figsize = (10, 8))
    plt.rcParams.update({'font.size': 30})
    ## this outputs a 1x1 subplot - change if you want more/less rows or columns ##
    ax = plt.gca()
    ax.tick_params(direction='in', length=6)
    ax.tick_params(axis='both', labelsize=20)
    ax.margins (0.5,0.5)
    plt.xlabel("SPV (V)", size = 20)
    ax.autoscale(enable=None, axis="x", tight=True)
    plt.ylabel("Distribution (V$^{-1}$)", fontsize=20)
    files=sorted([f for f in os.listdir(file_directory) if f.endswith('.txt')])
    n_files=len(files)
    colors=get_color_gradient(c1,c2,n_files)
    # Limit number of ticks
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))  # Max 5 ticks on x-axis
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))  # Max 5 ticks on y-axis
    # Loop through files in the specified directory
 #   for filename in os.listdir(file_directory):
    for idx, filename in enumerate(tqdm(files, desc="Processing files")):
        file_path = os.path.join(file_directory, filename)
        #print(f"\nProcessing file: {filename}")
        t += image_time
        time_scatter.append(t)

        with open(file_path, 'r', encoding='utf-8', errors='replace') as f:
            data = np.loadtxt(f, skiprows=rows)

        x_data = data[:, 0]
        y_data = data[:, 1]
            
            # Fit the Gaussian curve to this dataset
        A, mu, sigma, popt, pcov, min_raw, mean_raw, max_raw = fit_gaussian_from_dataset(x_data, y_data)
        min_scatter.append(min_raw)
        mean_scatter.append(mean_raw)
        max_scatter.append(max_raw)
        y_fit=gaussian(x_data,*popt)
        #for finding peak centre for fitted gaussian
        peak_index=np.argmax(y_fit)
        x_fit_max=x_data[peak_index]
        gaussian_scatter.append(x_fit_max)
        #Uncomment for debugging: 
        #print(f"Min:{min_scatter}\n Mean: {mean_scatter}\n Max: {max_scatter}")
        #print(time_scatter)
        # If fitting was successful, plot the result
        if A is not None:
            
            plt.scatter(x_data, y_data, label=f"{t}s", color=colors[idx], s=10)
            plt.plot(x_data, y_fit, label=f"_nolegend_", color=colors[idx], linewidth=2)
            plt.title(f"Single Gaussian Fitting", size=20)
            plt.xlabel("SPV (V)", size=20)
            plt.ylabel("Distribution $V^{-1}$", size=20)
            plt.autoscale()
            #plt.legend()
                
    print("All gaussians processed. Congrats!")
    # Add the colorbar after plotting
    cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
    norm = mcolors.Normalize(vmin=np.min(time_scatter), vmax=np.max(time_scatter))
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    fig.tight_layout()
    cbar = fig.colorbar(sm, ax=ax, orientation='vertical')
    cbar.set_label('Time (s)', size=20)
    cbar.ax.tick_params(labelsize=20)
    plt.autoscale(enable=True, axis='y', tight=False)
    #Comment out if no file saving required
    #fig=plt.gcf()
    #save_plot_to_folder(fig, save_path, filename=f'{filename}.jpeg')


    return min_scatter, mean_scatter, max_scatter, time_scatter, gaussian_scatter

    
    plt.show() #Keep after saving figure due to errors. 